Imports

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, random_split
from torch.utils.data import DataLoader
import torch.optim as optim
from sklearn.preprocessing import StandardScaler

import pandas as pd
 

Creating the NN

In [6]:
class Net(nn.Module):
    def __init__(self, d_in):
        super(Net, self).__init__()

        self.fc1 = nn.Linear(d_in, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, 32)
        self.fc4 = nn.Linear(32, 1)

    def forward(self, x):
        x = F.gelu(self.fc1(x))
        x = F.gelu(self.fc2(x))
        x = F.gelu(self.fc3(x))
        x = self.fc4(x)
        return x


Load The Heston Model Parameters

In [14]:
df_parquet_1 = pd.read_parquet("exotic_mc_dataset_1.parquet")
df_heston_selected_parquet_1 = df_parquet_1[df_parquet_1["model"] == "HESTON"].loc[:, "option":"T"].join(
    df_parquet_1.loc[df_parquet_1["model"] == "HESTON", "v0":"rho"]
)

df_parquet_2 = pd.read_parquet("exotic_mc_dataset_2.parquet")
df_heston_selected_parquet_2 = df_parquet_2[df_parquet_2["model"] == "HESTON"].loc[:, "option":"T"].join(
    df_parquet_2.loc[df_parquet_2["model"] == "HESTON", "v0":"rho"]
)

Y_parquet_1 = torch.tensor(df_heston_selected_parquet_1["price"].values, dtype=torch.float32).view(-1, 1)
Y_parquet_2 = torch.tensor(df_heston_selected_parquet_2["price"].values, dtype=torch.float32).view(-1, 1)

opt_dummies_1 = pd.get_dummies(
    df_heston_selected_parquet_1["option"],
    prefix="opt"
).astype("int64")

opt_dummies_2 = pd.get_dummies(
    df_heston_selected_parquet_2["option"],
    prefix="opt"
).astype("int64")

opt_dummies_2 = opt_dummies_2.reindex(
    columns=opt_dummies_1.columns,
    fill_value=0
)

num_cols = [
    "stderr",
    "S0",
    "K",
    "r",
    "T",
    "v0",
    "kappa",
    "theta",
    "xi",
    "rho"
]

X_df_1 = pd.concat(
    [
        df_heston_selected_parquet_1[num_cols],
        opt_dummies_1
    ],
    axis=1
)

X_df_2 = pd.concat(
    [
        df_heston_selected_parquet_2[num_cols],
        opt_dummies_2
    ],
    axis=1
)

X_parquet_1 = torch.tensor(X_df_1.values, dtype=torch.float32)
X_parquet_2 = torch.tensor(X_df_2.values, dtype=torch.float32)

X = torch.cat([X_parquet_1, X_parquet_2], dim=0)
Y = torch.cat([Y_parquet_1, Y_parquet_2], dim=0)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X.numpy())
X = torch.tensor(X_scaled, dtype=torch.float32)

input_dim = X.shape[1]
net = Net(input_dim)

display(X.shape)
display(Y.shape)

torch.Size([6618, 16])

torch.Size([6618, 1])

In [15]:
X_df_1

,stderr,S0,K,r,T,v0,kappa,theta,xi,rho,opt_ASIAN_ARITH_CALL,opt_ASIAN_ARITH_PUT,opt_BARRIER_DOWN_AND_OUT_PUT,opt_BARRIER_UP_AND_OUT_CALL,opt_LOOKBACK_FLOAT_CALL,opt_LOOKBACK_FLOAT_PUT
3,0.182873,102.485076,147.685871,0.027255,1.389348,0.494418,2.718432,0.364860,1.209757,-0.073835,0,1,0,0,0,0
6,0.000000,82.847837,112.842249,0.000005,0.548326,0.478771,3.801205,0.424584,0.376359,-0.600336,0,0,1,0,0,0
7,0.000000,63.109201,55.816582,0.019524,0.058221,0.366511,3.634015,0.398027,0.650938,-0.004666,0,0,1,0,0,0
10,0.000000,147.768757,93.497025,0.005463,0.646335,0.343687,3.230841,0.441155,1.707285,-0.725269,0,0,0,1,0,0
11,0.019087,103.091564,79.970279,0.040125,0.923979,0.139105,0.881485,0.163398,0.565200,-0.067560,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9992,0.000000,96.479731,122.885592,0.078665,0.196752,0.014711,0.584423,0.287987,0.721452,-0.642265,0,0,0,1,0,0
9994,0.299746,88.812921,62.188946,0.006207,1.817779,0.440693,2.061102,0.198395,0.961681,-0.131543,0,0,0,0,1,0
9995,0.046912,64.857144,147.328215,0.004579,0.107087,0.330328,0.501510,0.280286,0.292201,-0.022449,0,0,0,0,1,0
9996,0.111302,123.312902,130.558532,0.001917,0.316705,0.348388,3.492039,0.133702,1.415792,-0.444135,0,0,0,0,1,0


Setting the DataLoader                                    

In [16]:
dataset = TensorDataset(X, Y)

#Lets keep 70% and 30%
n_total = len(dataset)
n_train = int(0.7 * n_total)
n_test = n_total - n_train

train_dataset, test_dataset = random_split(dataset, [n_train, n_test])


train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=128, shuffle=False)
train_loader

Define a Loss function and optimizer

In [17]:
criterion = nn.MSELoss()
optimizer = optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-2)

Training the Network

In [18]:
for epoch in range(100):  # loop over the dataset multiple times

    running_loss = 0.0
    for i, data in enumerate(train_loader, 0):
        # get the inputs; data is a list of [inputs, labels]
        inputs, labels = data

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # print statistics
        running_loss += loss.item()
        if i % 20 == 19:    # print every 2000 mini-batches
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 20:.3f}')
            running_loss = 0.0

print('Finished Training')

[1,    20] loss: 630.224
[2,    20] loss: 541.076
[3,    20] loss: 177.241
[4,    20] loss: 135.848
[5,    20] loss: 95.314
[6,    20] loss: 60.352
[7,    20] loss: 49.105
[8,    20] loss: 38.952
[9,    20] loss: 28.627
[10,    20] loss: 21.321
[11,    20] loss: 21.776
[12,    20] loss: 21.398
[13,    20] loss: 17.955
[14,    20] loss: 17.120
[15,    20] loss: 14.259
[16,    20] loss: 15.089
[17,    20] loss: 12.822
[18,    20] loss: 12.228
[19,    20] loss: 13.235
[20,    20] loss: 11.360
[21,    20] loss: 10.871
[22,    20] loss: 9.428
[23,    20] loss: 11.897
[24,    20] loss: 9.904
[25,    20] loss: 11.115
[26,    20] loss: 9.920
[27,    20] loss: 9.606
[28,    20] loss: 9.434
[29,    20] loss: 8.875
[30,    20] loss: 9.150
[31,    20] loss: 9.077
[32,    20] loss: 7.602
[33,    20] loss: 7.823
[34,    20] loss: 7.842
[35,    20] loss: 7.688
[36,    20] loss: 7.412
[37,    20] loss: 6.874
[38,    20] loss: 7.674
[39,    20] loss: 7.224
[40,    20] loss: 7.153
[41,    20] loss: 6.88

In [19]:
net.eval()

correct = 0
total = 0
tol = 0.05  # absolute tolerance

with torch.no_grad():
    for X, y in test_loader:
        preds = net(X).view_as(y)              # ensure shape matches (N,1)
        within = (preds - y).abs() <= tol      # boolean tensor
        correct += within.sum().item()
        total += y.numel()

acc = 100.0 * correct / total
print(f"Accuracy within ±{tol}: {acc:.2f}%")

Accuracy within ±0.05: 7.30%


In [20]:
net.eval()

correct = 0
total = 0
pct = 0.20  # 20%

eps = 1e-8  # avoid divide-by-zero
with torch.no_grad():
    for X, y in test_loader:
        preds = net(X).view_as(y)
        rel_err = (preds - y).abs() / (y.abs() + eps)
        within = rel_err <= pct
        correct += within.sum().item()
        total += y.numel()

acc = 100.0 * correct / total
print(f"Accuracy within ±{pct*100:.1f}%: {acc:.2f}%")

Accuracy within ±20.0%: 53.47%
